# imports

In [28]:
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, accuracy_score
import mlflow
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
import mlflow.tensorflow
from tensorflow.keras.callbacks import EarlyStopping

# config

In [9]:
ASSET = "BTC"
INTERVAL = "1h"

In [14]:
TIMESTEPS = 24

# mlflow

In [10]:
mlflow.set_tracking_uri("http://localhost:5000")

experiment_name = f"{ASSET}_{INTERVAL}_deep_learning"
mlflow.set_experiment(experiment_name)

2026/05/10 12:59:38 INFO mlflow.tracking.fluent: Experiment with name 'BTC_1h_deep_learning' does not exist. Creating a new experiment.


<Experiment: artifact_location='/mlruns/13', creation_time=1778414378831, experiment_id='13', last_update_time=1778414378831, lifecycle_stage='active', name='BTC_1h_deep_learning', tags={}, trace_location=None, workspace='default'>

# load data

In [11]:
train_df = pd.read_parquet('../../../data/processed/train_btc_1h.parquet')
test_df = pd.read_parquet('../../../data/processed/test_btc_1h.parquet')

# splitting

In [12]:
features = [col for col in train_df.columns if col not in ['target_1h', 'target_direction']]
X_train = train_df[features]
y_train = train_df['target_direction']
X_test = test_df[features]
y_test = test_df['target_direction']

# checking the train data if scaled or not

In [13]:
print(X_train.describe().loc[['mean', 'std']].T.head(5))

                mean       std
open   -3.614590e-16  1.000012
high   -1.913607e-16  1.000012
low     0.000000e+00  1.000012
close   6.378689e-17  1.000012
volume  4.518238e-17  1.000012


# LSTM

In [16]:
def create_sequences(X, y, time_steps=24):
    """
    Transforms 2D tabular data into 3D tensors for LSTM consumption.
    X: Numpy array of features
    y: Target series
    time_steps: historical periods to look back
    """
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:(i + time_steps)])
        ys.append(y.iloc[i + time_steps])
        
    return np.array(Xs), np.array(ys)


X_train_seq, y_train_seq = create_sequences(X_train.values, y_train, TIMESTEPS)
X_test_seq, y_test_seq = create_sequences(X_test.values, y_test, TIMESTEPS)

print(f"Training Shape (Samples, TimeSteps, Features): {X_train_seq.shape}")
print(f"Testing Shape: {X_test_seq.shape}")

Training Shape (Samples, TimeSteps, Features): (42751, 24, 39)
Testing Shape: (10670, 24, 39)


# model

In [ ]:
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(TIMESTEPS, X_train_seq.shape[2])),
    Dropout(0.2),
    
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    
    Dense(16, activation='relu'),
    
    Dense(1, activation='sigmoid')
])

c:\Users\Manindra\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


# compile model

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [27]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 24, 64)         │        26,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 39,585 (154.63 KB)

 Trainable params: 39,585 (154.63 KB)

 Non-trainable params: 0 (0.00 B)

# mlflow tensorflow 

In [29]:
mlflow.tensorflow.autolog()

2026/05/10 13:23:40 WARNING mlflow.utils.autologging_utils: MLflow tensorflow autologging is known to be compatible with 2.16.2 <= tensorflow, but the installed version is 2.16.1. If you encounter errors during autologging, try upgrading / downgrading tensorflow to a compatible version, or try upgrading MLflow.


# early stopping

In [30]:
early_stopping = EarlyStopping(
    monitor='val_loss', 
    patience=5,
    restore_best_weights=True
)

# run, fit model with mlflow

In [32]:
with mlflow.start_run(run_name="LSTM_Baseline"):
    
    mlflow.log_param("time_steps", TIMESTEPS)
    mlflow.log_param("model_type", "LSTM_2_Layer")
    
    print("started training")
    
    history = model.fit(
        X_train_seq, y_train_seq,
        epochs=50,
        batch_size=64,
        validation_split=0.2,
        callbacks=[early_stopping],
        verbose=1
    )
    
    test_loss, test_acc = model.evaluate(X_test_seq, y_test_seq, verbose=0)
    
    mlflow.log_metric("test_loss", test_loss)
    mlflow.log_metric("test_accuracy", test_acc)
    
    print(f"\n--- Training Complete ---")
    print(f"Test Accuracy: {test_acc:.4f}")

started training


Epoch 1/50
533/535 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5073 - loss: 0.6928

535/535 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.5073 - loss: 0.6928 - val_accuracy: 0.5096 - val_loss: 0.6928
Epoch 2/50
535/535 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.5120 - loss: 0.6925 - val_accuracy: 0.5088 - val_loss: 0.6944
Epoch 3/50
535/535 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.5185 - loss: 0.6923 - val_accuracy: 0.5099 - val_loss: 0.6928
Epoch 4/50
535/535 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.5202 - loss: 0.6920 - val_accuracy: 0.5044 - val_loss: 0.6945
Epoch 5/50
535/535 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.5227 - loss: 0.6919 - val_accuracy: 0.5036 - val_loss: 0.6942
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step


2026/05/10 13:37:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/10 13:38:01 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during tensorflow autologging: 0.6927725672721863 is not in list



--- Training Complete ---
Test Accuracy: 0.5026
🏃 View run LSTM_Baseline at: http://localhost:5000/#/experiments/13/runs/6e9432beffea45939fdfb49cd1fb2d99
🧪 View experiment at: http://localhost:5000/#/experiments/13
